# Anàlisi de biaixos i debilitats del model

Consumeix `outputs/predictions.csv` generat per `python analyze.py`.

**Seccions:**
- A — Biaixos lingüístics i col·lapse de diversitat
- B — Casos pitjors i millors (inspecció visual)
- C — Errors per categoria visual
- D — Biaix de gènere

In [ ]:
import os
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

CSV_PATH = '../outputs/predictions.csv'
IMAGES_DIR = '../data/flickr8k/Images'

df = pd.read_csv(CSV_PATH)
print(f'Rows: {len(df)}')
df.head()

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z']+", str(text).lower())

def ngrams(tokens, n):
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

df['pred_tokens'] = df['predicted_caption'].apply(tokenize)
df['ref_tokens_flat'] = df['references'].apply(lambda s: tokenize(str(s).replace('|||', ' ')))

## A — Biaixos lingüístics i col·lapse de diversitat

In [ ]:
pred_tokens_all = [t for toks in df['pred_tokens'] for t in toks]
ref_tokens_all = [t for toks in df['ref_tokens_flat'] for t in toks]

ttr_pred = len(set(pred_tokens_all)) / max(len(pred_tokens_all), 1)
ttr_ref = len(set(ref_tokens_all)) / max(len(ref_tokens_all), 1)
uniq_preds = df['predicted_caption'].nunique()

print(f'Type-Token Ratio prediccions : {ttr_pred:.4f}  (vocab={len(set(pred_tokens_all))})')
print(f'Type-Token Ratio referències : {ttr_ref:.4f}  (vocab={len(set(ref_tokens_all))})')
print(f'Captions úniques predites    : {uniq_preds} / {len(df)} ({100*uniq_preds/len(df):.1f}%)')

In [ ]:
def top_ngrams(token_lists, n, k=20):
    c = Counter()
    for toks in token_lists:
        c.update(ngrams(toks, n))
    return c.most_common(k)

for n in (1, 2, 3):
    print(f'\n== Top {n}-grams ==')
    pred_top = top_ngrams(df['pred_tokens'], n, 15)
    ref_top = top_ngrams(df['ref_tokens_flat'], n, 15)
    print(f'Prediccions: {pred_top}')
    print(f'Referències: {ref_top}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['pred_length'], bins=20, alpha=0.7, label='predicció')
axes[0].hist(df['ref_avg_length'], bins=20, alpha=0.7, label='referència (mitjana)')
axes[0].set_xlabel('Longitud (tokens)'); axes[0].set_ylabel('# imatges'); axes[0].legend()
axes[0].set_title('Distribució de longituds')

axes[1].hist(df['bleu_1'], bins=30)
axes[1].set_xlabel('BLEU-1 per mostra'); axes[1].set_ylabel('# imatges')
axes[1].set_title('Distribució de BLEU-1')
plt.tight_layout(); plt.show()

## B — Casos pitjors i millors

In [ ]:
def show_examples(subset, title, n=6):
    print(f'\n=== {title} ===')
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    for ax, (_, row) in zip(axes, subset.head(n).iterrows()):
        path = os.path.join(IMAGES_DIR, row['image_name'])
        try:
            ax.imshow(Image.open(path))
        except Exception as e:
            ax.text(0.5, 0.5, f'(no img)\n{e}', ha='center')
        ax.axis('off')
        title_txt = f"BLEU-1={row['bleu_1']:.2f} CLIP={row['clip_score']:.1f}\nPred: {row['predicted_caption'][:60]}"
        ax.set_title(title_txt, fontsize=8)
    plt.tight_layout(); plt.show()
    for _, row in subset.head(n).iterrows():
        print(f"[{row['image_name']}]")
        print(f"  Pred: {row['predicted_caption']}")
        print(f"  Refs: {row['references'][:200]}")

show_examples(df.sort_values('bleu_1').head(6), 'Pitjors per BLEU-1')
show_examples(df.sort_values('bleu_1', ascending=False).head(6), 'Millors per BLEU-1')
show_examples(df.sort_values('clip_score').head(6), 'Pitjors per CLIPScore')
show_examples(df.sort_values('clip_score', ascending=False).head(6), 'Millors per CLIPScore')


## C — Errors per categoria visual

Agrupació per paraules clau a les referències.

In [ ]:
CATEGORIES = {
    'dog':    {'dog', 'dogs', 'puppy', 'puppies'},
    'child':  {'child', 'children', 'kid', 'kids', 'boy', 'girl', 'baby'},
    'water':  {'water', 'beach', 'pool', 'lake', 'ocean', 'sea', 'river', 'swim', 'swimming'},
    'snow':   {'snow', 'snowy', 'ski', 'skiing', 'snowboard'},
    'sport':  {'soccer', 'football', 'ball', 'basketball', 'tennis', 'baseball', 'sport'},
    'bike':   {'bike', 'biking', 'bicycle', 'motorcycle'},
    'grass':  {'grass', 'field', 'park'},
    'street': {'street', 'road', 'sidewalk', 'city'},
}

def categorize(ref_tokens):
    s = set(ref_tokens)
    return [name for name, vocab in CATEGORIES.items() if s & vocab]

df['categories'] = df['ref_tokens_flat'].apply(categorize)

# Corpus BLEU per categoria (alineat amb train.py:evaluate)
from nltk.translate.bleu_score import corpus_bleu
_smooth = SmoothingFunction().method1

def corpus_bleu_n(sub, n):
    if len(sub) < 5: return None
    hyps = [tokenize(p) for p in sub['predicted_caption']]
    refs = [[tokenize(r) for r in str(rs).split(' ||| ')] for rs in sub['references']]
    weights = tuple([1/n]*n + [0]*(4-n))
    return corpus_bleu(refs, hyps, weights=weights, smoothing_function=_smooth)

rows = []
for cat in CATEGORIES:
    mask = df['categories'].apply(lambda cs: cat in cs)
    if mask.sum() < 5:
        continue
    sub = df.loc[mask]
    rows.append({
        'category': cat,
        'n': int(mask.sum()),
        'bleu_1': corpus_bleu_n(sub, 1),
        'bleu_4': corpus_bleu_n(sub, 4),
        'clip_score': sub['clip_score'].mean(),
    })
summary = pd.DataFrame(rows).sort_values('bleu_1')
print('Categories ordenades per BLEU-1 corpus (pitjor → millor):')
summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(summary))
ax.bar(x - 0.2, summary['bleu_1'], 0.4, label='BLEU-1')
ax.bar(x + 0.2, summary['clip_score'] / 100, 0.4, label='CLIPScore/100')
ax.set_xticks(x); ax.set_xticklabels(summary['category'], rotation=30)
ax.legend(); ax.set_title('Rendiment per categoria visual')
plt.tight_layout(); plt.show()

## D — Biaix de gènere

Quan la referència parla d'home/dona, què prediu el model?

In [ ]:
def ref_gender(row):
    m, w = row['refs_mention_man'], row['refs_mention_woman']
    if m and not w: return 'man'
    if w and not m: return 'woman'
    if m and w:     return 'both'
    return 'neutral'

def pred_gender(row):
    m, w = row['pred_mentions_man'], row['pred_mentions_woman']
    if m and not w: return 'man'
    if w and not m: return 'woman'
    if m and w:     return 'both'
    return 'neutral'

df['ref_gender'] = df.apply(ref_gender, axis=1)
df['pred_gender'] = df.apply(pred_gender, axis=1)

confusion = pd.crosstab(df['ref_gender'], df['pred_gender'], margins=True)
print('Matriu de confusió de gènere (files=referència, columnes=predicció):')
print(confusion)

In [ ]:
# Casos clau: referència diu dona però el model prediu home (i viceversa)
swap_w_to_m = df[(df['ref_gender'] == 'woman') & (df['pred_gender'] == 'man')]
swap_m_to_w = df[(df['ref_gender'] == 'man') & (df['pred_gender'] == 'woman')]
neutral_gendered = df[(df['ref_gender'] == 'neutral') & (df['pred_gender'].isin(['man', 'woman']))]

print(f'Swap dona→home (refs diuen dona, predicció diu home): {len(swap_w_to_m)}')
print(f'Swap home→dona (refs diuen home, predicció diu dona): {len(swap_m_to_w)}')
print(f'Genderització de referències neutres: {len(neutral_gendered)}')
print(f'  → de les quals prediuen home: {(neutral_gendered["pred_gender"] == "man").sum()}')
print(f'  → de les quals prediuen dona: {(neutral_gendered["pred_gender"] == "woman").sum()}')

if len(swap_w_to_m) > 0:
    show_examples(swap_w_to_m, 'Refs=dona, Pred=home', n=min(30, len(swap_w_to_m)))
# Dump per comparativa entre variants a comparison.ipynb
os.makedirs(os.path.dirname(CSV_PATH) or '.', exist_ok=True)
_stem = os.path.splitext(os.path.basename(CSV_PATH))[0].replace('predictions_', '').replace('predictions', 'v1')
swap_w_to_m[['image_name', 'predicted_caption', 'references']].to_csv(f'../outputs/swaps_w_to_m_{_stem}.csv', index=False)
print(f'Saved → ../outputs/swaps_w_to_m_{_stem}.csv ({len(swap_w_to_m)} files)')


## Conclusions accionables

_Omple aquesta secció després d'executar les anteriors:_

1. **Diversitat lèxica:** TTR predicció vs referència, % de prediccions úniques → indica mode collapse?
2. **N-grames sobre-representats:** quins són els clixés del model? Suggereix re-pesar el loss per freqüència inversa.
3. **Categories febles:** on cau més el rendiment? Oversampling dirigit durant fine-tuning.
4. **Biaix de gènere:** quin tipus de swap és més freqüent? Quantes referències neutres es generen amb gènere?

### Pla de fine-tuning correctiu

Segons les troballes anteriors, prioritzar (només una o dues, no totes alhora):

- **Mitigar mode collapse:** loss ponderat per freqüència inversa d'n-grames, o decoding amb nucleus sampling (no requereix re-entrenar).
- **Reequilibrar categories visuals:** oversampling de les categories amb pitjor BLEU/CLIP al train set.
- **Reduir biaix de gènere:** penalitzar prediccions amb gènere quan totes les referències del train són neutres (loss auxiliar), o filtrar mostres ambigües del train.
- **Mitigar overfitting general:** més weight decay, label smoothing més agressiu, early stopping per CLIPScore.